# 01 – Exploratory Data Analysis

This notebook loads the synthetic credit card transaction dataset and performs exploratory analysis including:
- Dataset overview and shape
- Class distribution (fraud vs normal)
- Amount distributions per class
- Time-based transaction patterns
- Feature correlation heatmap
- Summary statistics

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Ensure the data generator is importable
sys.path.insert(0, os.path.join('..', 'data'))
from generate_synthetic_data import generate_synthetic_data

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Dataset

In [ ]:
DATA_PATH = os.path.join('..', 'data', 'creditcard.csv')

if not os.path.exists(DATA_PATH):
    print('Generating synthetic dataset…')
    df = generate_synthetic_data(n_samples=100_000, fraud_ratio=0.02)
    df.to_csv(DATA_PATH, index=False)
    print(f'Saved to {DATA_PATH}')
else:
    df = pd.read_csv(DATA_PATH)

print(f'Shape: {df.shape}')
df.head()

## 2. Class Distribution

In [ ]:
class_counts = df['Class'].value_counts()
print('Class distribution:')
print(class_counts)
print(f'\nFraud ratio: {class_counts[1] / len(df) * 100:.2f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
class_counts.plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'])
axes[0].set_title('Transaction Class Counts')
axes[0].set_xlabel('Class (0=Normal, 1=Fraud)')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Pie chart
axes[1].pie(class_counts, labels=['Normal', 'Fraud'], autopct='%1.2f%%',
            colors=['steelblue', 'tomato'], startangle=90)
axes[1].set_title('Class Distribution')

plt.tight_layout()
plt.show()

## 3. Amount Distribution by Class

In [ ]:
normal = df[df['Class'] == 0]
fraud  = df[df['Class'] == 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(normal['Amount'], bins=50, color='steelblue', alpha=0.7, log=True)
axes[0].set_title('Normal Transaction Amounts')
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Count (log scale)')

axes[1].hist(fraud['Amount'], bins=50, color='tomato', alpha=0.7, log=True)
axes[1].set_title('Fraudulent Transaction Amounts')
axes[1].set_xlabel('Amount ($)')
axes[1].set_ylabel('Count (log scale)')

plt.tight_layout()
plt.show()

print('Normal amount stats:')
print(normal['Amount'].describe())
print('\nFraud amount stats:')
print(fraud['Amount'].describe())

## 4. Time-Based Patterns

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(normal['Time'], normal['Amount'], alpha=0.1, s=2, c='steelblue', label='Normal')
ax.scatter(fraud['Time'],  fraud['Amount'],  alpha=0.5, s=10, c='tomato',    label='Fraud')
ax.set_title('Transactions Over Time')
ax.set_xlabel('Time (seconds)')
ax.set_ylabel('Amount ($)')
ax.legend()
plt.show()

## 5. Correlation Matrix (sample of V features)

In [ ]:
v_cols = [f'V{i}' for i in range(1, 11)] + ['Amount', 'Class']
corr = df[v_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, square=True)
plt.title('Feature Correlation Matrix (V1-V10, Amount, Class)')
plt.tight_layout()
plt.show()

## 6. Summary Statistics

In [ ]:
print('Overall summary statistics:')
display(df.describe())